# Sturm Manifest Builder v3.3

**Proyecto:** Inferencia del envejecimiento mediante modelo latente multimodal  
**Propósito:** Construir un manifest reproducible que una imágenes brightfield con el CSV central de biomarcadores (Lifespan_Study_Data.csv) y los metadatos de GEO (RNA-seq, metilación).  




El notebook construye un **manifest multimodal reproducible** para el proyecto de envejecimiento celular, uniendo imágenes brightfield con el CSV central y metadata de GEO.

### Flujo principal
1. **Configuración de rutas**: define paths de CSV, imágenes, GEO y salida.
2. **Carga y estandarización del CSV central**:
   - Renombra columnas clave.
   - Normaliza `cell_line`, `treatment`, tipos numéricos y PDL.
   - Verifica unicidad de llaves.
3. **Ingeniería de PDL**:
   - Calcula `pdl_norm` por línea celular.
   - Crea bins `early/mid/late`.
4. **Parsing de imágenes por fase (I, II, III, V)**:
   - Regex específicos por formato de nombre.
   - Tests internos de parser.
5. **Escaneo de imágenes + diagnóstico de fallas**:
   - Extrae metadatos de cada archivo.
   - Registra exclusiones y errores de parseo.
6. **Join imágenes ↔ CSV**:
   - Normalización extendida de treatments.
   - Join principal por llaves biológicas.
   - Manejo especial de Contact Inhibition.
   - Flags de modalidades (`has_rna`, `has_methylation`, etc.).
7. **Integración GEO (RNA-seq y metilación)**:
   - Join por claves normalizadas para anexar metadata GEO.
8. **Splits anti-leakage por `cell_line`**:
   - Hay versión con GroupKFold y otra con asignación manual de 3 folds.
9. **Validación/contrato de integridad**:
   - Asserts de unicidad, cobertura PDL, columnas requeridas, coherencia de folds.
10. **Export**:
   - Guarda manifest(s) versionados con timestamp + MD5 + JSON de metadatos.
11. **Subconjuntos MVP-1**:
   - Genera `pretrain`, `finetune`, `eval` con criterios biológicos y de calidad.


## 0. CONFIG — Ajusta estas 5 rutas antes de correr

In [12]:
from pathlib import Path

# ─── AJUSTA ESTAS RUTAS ────────────────────────────────────────────────────────
CSV_PATH         = Path(r'/Users/JCB/Documentos/Proyecto Integrador/data/Lifespan_Study_Data.csv')
IMAGES_ROOT      = Path(r'/Volumes/SanDisk SSD 1TB/Storage/Data/Cellular_Lifespan_Study_Brightfield_Images')
GEO_RNA_META     = Path(r'/Users/JCB/Documentos/Proyecto Integrador/data/GSE179848_sample_metadata.csv')
GEO_METH_META    = Path(r'/Users/JCB/Documentos/Proyecto Integrador/data/GSE179847_sample_metadata.csv')
OUTPUT_DIR       = Path(r'/Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS')
# ──────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Config OK')
print(f'  CSV:        {CSV_PATH}')
print(f'  Imágenes:   {IMAGES_ROOT}')
print(f'  GEO RNA:    {GEO_RNA_META}')
print(f'  GEO Meth:   {GEO_METH_META}')
print(f'  Output:     {OUTPUT_DIR}')

Config OK
  CSV:        /Users/JCB/Documentos/Proyecto Integrador/data/Lifespan_Study_Data.csv
  Imágenes:   /Volumes/SanDisk SSD 1TB/Storage/Data/Cellular_Lifespan_Study_Brightfield_Images
  GEO RNA:    /Users/JCB/Documentos/Proyecto Integrador/data/GSE179848_sample_metadata.csv
  GEO Meth:   /Users/JCB/Documentos/Proyecto Integrador/data/GSE179847_sample_metadata.csv
  Output:     /Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS


## 1. Imports y utilidades

In [13]:
import re
import hashlib
import datetime
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupKFold

# ── Normalización de cell_line ────────────────────────────────────────────────
def normalize_cell_line(raw: str) -> str:
    if pd.isna(raw):
        return None
    s = str(raw).strip()
    m = re.match(r'(hFB(\d+))(?:-([mf]))?', s, re.IGNORECASE)
    if m:
        return f'hFB{int(m.group(2))}'
    return s.lower()

def extract_sex_suffix(raw: str) -> str:
    if pd.isna(raw):
        return None
    m = re.search(r'hFB\d+-([mf])', str(raw), re.IGNORECASE)
    return m.group(1).lower() if m else None

# ── Normalización de treatments — función canónica del CSV ────────────────────
# Mapea al valor exacto que usa el CSV de Sturm en columna 'Treatments'
TREATMENT_MAP = {
    'ctrl': 'Control', 'control': 'Control', 'contorl': 'Control',
    'dex': 'DEX', 'acute dex': 'AcuteDEX',
    'mitoq': 'MitoQ', 'mitoq + dex': 'MitoQ+DEX', 'mitoq+dex': 'MitoQ+DEX',
    'nac': 'NAC', 'nac + dex': 'NAC+DEX', 'nac+dex': 'NAC+DEX',
    'puls': 'Pulsated_DEX',
    # AKG → usar el typo del CSV de Sturm ('a-ketogluturate' con u)
    'a-keto': 'a-ketogluturate',
    'a-ketoglutarate': 'a-ketogluturate',
    'mod': 'Oligomycin', 'modulators': 'Oligomycin',
    'mod+dex': 'Oligomycin+DEX', 'modulators+dex': 'Oligomycin+DEX',
    'oligo': 'Oligomycin', 'oligomycin': 'Oligomycin',
    'oligo+dex': 'Oligomycin+DEX', 'oligomycin+dex': 'Oligomycin+DEX',
    '5-azacytidine': '5-Azacytidine',
    'contact inhibition': 'Contact_Inhibition',
    'contact inhibtion': 'Contact_Inhibition',
    'arrival': 'Control', 'vehicle': 'Control', 'post-thaw': 'Control',
}

def normalize_treatment(raw: str) -> str:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return 'Control'
    key = str(raw).strip().lower()
    return TREATMENT_MAP.get(key, str(raw).strip())

print('Utilidades cargadas OK')

Utilidades cargadas OK


## 2. Cargar y estandarizar el CSV central
El CSV tiene shape (1919, 271). Columnas clave identificadas en inspección previa.

In [14]:
# ══ SECCIÓN 2 — Cargar y estandarizar el CSV central ════════════════════════

df_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f'CSV shape: {df_raw.shape}')

COL_MAP = {
    'Sample':                    'sample_id',
    'Cell_Line':                 'cell_line_raw',
    'Passage':                   'passage',
    'Study_Part':                'study_part',
    'Percent_Oxygen':            'percent_oxygen',
    'Treatments':                'treatment_clean',
    'Treatment':                 'treatment_full',
    'Population_Doublings_DI':   'pdl',
    'Percent_Lifespan':          'percent_lifespan',
    'Clinical_Condition':        'clinical_condition',
    'Donor_Age':                 'donor_age',
    'Cell_Type':                 'cell_type',
    'Days_Grown':                'days_grown',
    'Time_Point':                'time_point',
    'Telomere_Length':           'telomere_length',
    'Copy_Number':               'mtdna_cn',
    'Cell_Size':                 'cell_size',
    'Cell_Volume':               'cell_volume',
    'Baseline_Respiration':      'seahorse_respiration',
    'Baseline_ECAR':             'seahorse_ecar',
    'RNAseq_ID':                 'rnaseq_id',
    'basename':                  'methylation_basename',
    'Horvath1':                  'clock_horvath1',
    'PhenoAge':                  'clock_phenoage',
    'GrimAge':                   'clock_grimage',
    'Mitotic_Age':               'clock_mitotic',
    'DNAmTL':                    'clock_dnamtl',
    'DNAmSen':                   'clock_dnamsen',
    'DunedinPoAm_38':            'clock_dunedin',
}

existing_cols = {k: v for k, v in COL_MAP.items() if k in df_raw.columns}
missing_cols  = {k: v for k, v in COL_MAP.items() if k not in df_raw.columns}
if missing_cols:
    print(f'⚠️  Columnas no encontradas: {list(missing_cols.keys())}')

df = df_raw[list(existing_cols.keys())].rename(columns=existing_cols).copy()

df['cell_line']      = df['cell_line_raw'].apply(normalize_cell_line)
df['cell_line_sex']  = df['cell_line_raw'].apply(extract_sex_suffix)
df['passage']        = pd.to_numeric(df['passage'], errors='coerce')
df['pdl']            = pd.to_numeric(df.get('pdl', pd.Series(np.nan, index=df.index)), errors='coerce')
df['study_part']     = pd.to_numeric(df.get('study_part', pd.Series(np.nan, index=df.index)), errors='coerce').astype('Int64')
df['percent_oxygen'] = pd.to_numeric(df.get('percent_oxygen', pd.Series(np.nan, index=df.index)), errors='coerce').astype('Int64')
df['time_point']     = pd.to_numeric(df.get('time_point', pd.Series(np.nan, index=df.index)), errors='coerce').astype('Int64')
df['treatment_norm'] = df['treatment_clean'].apply(normalize_treatment)

print(f'\nDF estandarizado: {df.shape}')
print(f'sample_id únicos: {df["sample_id"].nunique()} / {len(df)}')
print(f'Cell lines: {sorted(df["cell_line"].dropna().unique())}')
print(f'Study_Parts: {sorted(df["study_part"].dropna().unique())}')
print(f'Percent_Oxygen: {sorted(df["percent_oxygen"].dropna().unique())}')
print(f'PDL range: {df["pdl"].min():.1f} – {df["pdl"].max():.1f}')
print(f'\nTreatments normalizados en CSV:')
print(sorted(df['treatment_norm'].dropna().unique()))
print(f'\nDisponibilidad de modalidades:')
for col, name in [('rnaseq_id','RNA-seq'), ('methylation_basename','Metilación'),
                  ('telomere_length','Telómero'), ('mtdna_cn','mtDNA CN')]:
    if col in df.columns:
        n = df[col].notna().sum()
        print(f'  {name:20s}: {n:4d} / {len(df)} ({100*n/len(df):.0f}%)')

CSV shape: (1919, 271)

DF estandarizado: (1919, 32)
sample_id únicos: 1919 / 1919
Cell lines: ['hFB1', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2', 'hFB6', 'hFB7', 'hFB8', 'hek293']
Study_Parts: [1, 2, 3, 4, 5]
Percent_Oxygen: [3, 21]
PDL range: 0.0 – 346.7

Treatments normalizados en CSV:
['2DG', '5-Azacytidine', '5-azacytidine+mitoNUITs', 'Contact_Inhibition', 'Contact_Inhibition_Regrowth', 'Control', 'DEX', 'Galactose', 'MitoQ', 'MitoQ+DEX', 'NAC', 'NAC+DEX', 'Oligomycin', 'Oligomycin+DEX', 'Pulsated_DEX', 'a-ketogluturate', 'betahydroxybutyrate', 'mitoNUITs', 'mitoNUITs+DEX']

Disponibilidad de modalidades:
  RNA-seq             :  345 / 1919 (18%)
  Metilación          :  496 / 1919 (26%)
  Telómero            :  496 / 1919 (26%)
  mtDNA CN            :  493 / 1919 (26%)


## 3. PDL normalizado por cell_line + bins early/mid/late

In [15]:
# ══ SECCIÓN 3 — PDL normalizado por cell_line + bins early/mid/late ═════════

df['pdl_norm'] = df.groupby('cell_line', group_keys=False)['pdl'].transform(
    lambda g: (g - g.min()) / (g.max() - g.min() + 1e-9)
)

def assign_bin(val, q33, q66):
    if pd.isna(val): return 'unknown'
    if val <= q33:   return 'early'
    if val <= q66:   return 'mid'
    return 'late'

pdl_bins = []
for cl, group in df.groupby('cell_line'):
    valid = group['pdl_norm'].dropna()
    q33 = valid.quantile(0.333) if len(valid) >= 3 else 0.333
    q66 = valid.quantile(0.666) if len(valid) >= 3 else 0.666
    bins = group['pdl_norm'].apply(lambda v: assign_bin(v, q33, q66))
    pdl_bins.append(bins)

df['pdl_bin'] = pd.concat(pdl_bins).reindex(df.index)

print('PDL normalizado OK')
print(df.groupby('cell_line')[['pdl','pdl_norm']].agg(['min','max','count']).round(2))
print('\nDistribución de bins:')
print(df.groupby(['cell_line','pdl_bin']).size().unstack(fill_value=0))

PDL normalizado OK
           pdl               pdl_norm           
           min     max count      min  max count
cell_line                                       
hFB1       0.0   60.36   218      0.0  1.0   218
hFB11      0.0   37.86    63      0.0  1.0    63
hFB12      0.0   68.45   496      0.0  1.0   496
hFB13      0.0   80.84   482      0.0  1.0   482
hFB14      0.0   44.94   213      0.0  1.0   213
hFB2       0.0   34.00   120      0.0  1.0   120
hFB6       0.0   25.23    68      0.0  1.0    68
hFB7       0.0   36.70    76      0.0  1.0    76
hFB8       0.0   34.30   109      0.0  1.0   109
hek293     0.0  346.67    55      0.0  1.0    55

Distribución de bins:
pdl_bin    early  late  mid  unknown
cell_line                           
hFB1          73    73   72        0
hFB11         21    21   21        0
hFB12        165   166  165        0
hFB13        161   161  160        1
hFB14         71    71   71        0
hFB2          40    40   40        0
hFB6          23    23   

## 4. Parser de imágenes — 4 parsers por fase

### Patrones identificados:
| Fase | Separador | Formato |
|------|-----------|---------|
| PhaseI  | espacios  | `hFB1-m P10  t175 N days growth 10x, Treatment` |
| PhaseII | comas     | `hFB12, Treatment, P6, 500kcells, t175, N-days growth, 10x` |
| PhaseIII| comas     | `hFB12, Treatment, P6, 500kcells, N-days growth, 21%, 10x` |
| PhaseV  | guiones   | `2021-02-24_ASM-01_hFB8_p35_3Ox_20x` |

In [16]:
# ══ SECCIÓN 4 — Parsers de imágenes v5 ══════════════════════════════════════

# ── Carpetas a excluir ────────────────────────────────────────────────────────
EXCLUDE_PATTERNS = [
    'other_treatments', 'pre-study', 'Countess', 'Contamination',
    'Contact Inhibtion', 'Contact Inhibition',
]

def should_exclude(path: Path) -> bool:
    # Excluir por carpeta
    if any(excl in path.parts for excl in EXCLUDE_PATTERNS):
        return True
    # FIX: excluir por contenido del filename (post-thaw y HEK293 están en el stem, no en carpetas)
    stem_lower = path.stem.lower()
    if stem_lower.startswith('hek293'):
        return True
    if 'post-thaw' in stem_lower:
        return True
    return False

# ── Patrones centralizados ────────────────────────────────────────────────────
DENSITY = r'[\d\.,\-]+[kKmM]cells'
MAG_END = r'(\d+)x(?:\s*#\d+)?$'

# ── PhaseI ────────────────────────────────────────────────────────────────────
RE_P1         = re.compile(r'^(hFB\d+(?:-[mf])?)\s+P(\d+)\s+[tT](\d+)\s+\d+\s+days?\s+growth\s+(\d+)x,?\s*(.*)$', re.IGNORECASE)
RE_P1_T_END   = re.compile(r'^(hFB\d+(?:-[mf])?)\s+P(\d+)\s+\d+\s+days?\s+growth\s+[tT](\d+)\s+(\d+)x\s*(.*)$', re.IGNORECASE)
RE_P1_SHORT   = re.compile(r'^(hFB\d+(?:-[mf])?)\s+P(\d+)\s+\d+\s+days?\s+growth\s+(\d+)x\s+(.+)$', re.IGNORECASE)
RE_P1_MAG_FIRST = re.compile(r'^(hFB\d+(?:-[mf])?)\s+P(\d+)\s+(\d+)x\s+\d+\s+days?\s+growth\s+[tT](\d+)$', re.IGNORECASE)
RE_P1_INV     = re.compile(r'^(hFB\d+(?:-[mf])?)\s+P(\d+)\s+(\d+)x\s+[tT](\d+)\s+\d+\s+days?\s+growth$', re.IGNORECASE)
RE_P1_MINIMAL = re.compile(r'^(hFB\d+(?:-[mf])?)\s+P(\d+)\s+(\d+)x\s+[tT](\d+)$', re.IGNORECASE)

# ── PhaseII ───────────────────────────────────────────────────────────────────
RE_P2_TREAT        = re.compile(rf'^(hFB\d+(?:-[mf])?),\s*([^,P][^,]*),\s*P(\d+),\s*({DENSITY}),\s*t(\d+),\s*[\d]+-days?\s+growth,?\s*{MAG_END}', re.IGNORECASE)
RE_P2_NOTREAT      = re.compile(rf'^(hFB\d+(?:-[mf])?),\s*P(\d+),\s*({DENSITY}),\s*t(\d+),\s*[\d]+-days?\s+growth,?\s*{MAG_END}', re.IGNORECASE)
RE_P2_NO_FLASK     = re.compile(rf'^(hFB\d+(?:-[mf])?),\s*([^,]+),\s*P(\d+),\s*({DENSITY}),\s*[\d]+-days?\s+growth,\s*{MAG_END}', re.IGNORECASE)
RE_P2_EXTRA        = re.compile(rf'^(hFB\d+(?:-[mf])?),\s*([^,]+),\s*P(\d+),\s*({DENSITY}),\s*[\d]+-days?\s+growth,\s*[^,\d]+,\s*(\d+)x(?:\s*#\d+)?(?:,.*)?$', re.IGNORECASE)
RE_P2_EXTRA_AFTER  = re.compile(rf'^(hFB\d+(?:-[mf])?),\s*([^,]+),\s*P(\d+),\s*({DENSITY}),\s*[\d]+-days?\s+growth,\s*(\d+)x(?:\s*#\d+)?,.*$', re.IGNORECASE)
RE_P2_SUFFIX_NC    = re.compile(rf'^(hFB\d+(?:-[mf])?),\s*([^,]+),\s*P(\d+),\s*({DENSITY}),\s*[\d]+-days?\s+growth,\s*(\d+)x\s+\w[\w\s]*$', re.IGNORECASE)
RE_P2_UNKNOWN_DEN  = re.compile(r'^(hFB\d+(?:-[mf])?),\s*([^,]+),\s*P(\d+),\s*unknown,\s*[tT](\d+),\s*[\d]+-days?\s+growth,\s*(\d+)x(?:\s*#\d+)?$', re.IGNORECASE)
RE_P2_TREAT_NOSP   = re.compile(rf'^(hFB\d+(?:-[mf])?),\s*([A-Za-z+\-]+)\s+P(\d+),\s*({DENSITY}),\s*[tT](\d+),\s*[\d]+-days?\s+growth,\s*(\d+)x(?:\s*#\d+)?$', re.IGNORECASE)
RE_P2_DEN_PAREN    = re.compile(r'^(hFB\d+(?:-[mf])?),\s*([^,]+),\s*P(\d+),\s*([\d\.,\-]+[kKmM]cells)\([^)]+\),\s*[\d]+-days?\s+growth,\s*(\d+)x(?:\s*#\d+)?$', re.IGNORECASE)

# ── PhaseIII ──────────────────────────────────────────────────────────────────
RE_P3        = re.compile(r'^(hFB\d+(?:-[mf])?),\s*([^,]+),\s*P(\d+),\s*([^,]+cells),\s*[\d]+-days?\s+growth,\s*(\d+)%,\s*(\d+)x$', re.IGNORECASE)
RE_P3_TPOINT = re.compile(r'^(hFB\d+(?:-[mf])?),\s*([^,]+),\s*P(\d+),\s*([^,]+cells),\s*[\d]+-days?\s+growth,\s*(\d+)%,\s*T\d+,\s*(\d+)x$', re.IGNORECASE)
RE_P3_NOO2   = re.compile(r'^(hFB\d+(?:-[mf])?),\s*([^,]+),\s*P(\d+),\s*([^,]+cells),\s*[\d]+-days?\s+growth,\s*(\d+)x(?:,.*)?$', re.IGNORECASE)

# ── PhaseV ────────────────────────────────────────────────────────────────────
RE_P5          = re.compile(r'^(\d{4}-\d{2}-\d{2})_(ASM-\d+)_(hFB\d+(?:-[mf])?)_p(\d+)_(\d+)Ox_(\d+)x$', re.IGNORECASE)
RE_P5_DUP      = re.compile(r'^(\d{4}-\d{2}-\d{2})_(ASM-\d+)_(hFB\d+(?:-[mf])?)_p(\d+)_(\d+)Ox_(\d+)x-\d+$', re.IGNORECASE)
RE_P5_SUFFIX   = re.compile(r'^(\d{4}-\d{2}-\d{2})_(ASM-\d+)_(hFB\d+(?:-[mf])?)_p(\d+)_(\d+)Ox_(\d+)x[\d]*_.+$', re.IGNORECASE)
RE_P5_DASH     = re.compile(r'^(\d{4}-\d{2}-\d{2})_(ASM-\d+)_(hFB\d+(?:-[mf])?)_p(\d+)_(\d+)Ox_(\d+)x-.+$', re.IGNORECASE)
RE_P5_DECIMAL  = re.compile(r'^(\d{4}-\d{2}-\d{2})_(ASM-\d+)_(hFB\d+(?:-[mf])?)_p(\d+)_(\d+)Ox_(\d+)\.(\d+)x$', re.IGNORECASE)

def _p5_result(m, mag_override=None):
    return {
        'cell_line_raw': m.group(3), 'treatment_raw': 'Control',
        'passage': int(m.group(4)), 'flask_size': None,
        'magnification': mag_override if mag_override is not None else int(m.group(6)),
        'o2_percent': int(m.group(5)), 'seeding_density': None,
        'microscope': m.group(2), 'image_date': m.group(1),
    }

def parse_image_filename(stem: str, phase: str) -> dict | None:

    if phase == 'PhaseI':
        for pattern in [RE_P1, RE_P1_T_END, RE_P1_SHORT, RE_P1_INV, RE_P1_MAG_FIRST, RE_P1_MINIMAL]:
            m = pattern.match(stem)
            if not m: continue
            g = m.groups()
            if pattern == RE_P1:
                cl, pas, flask, mag, treat = g[0], g[1], g[2], g[3], g[4]
            elif pattern == RE_P1_T_END:
                cl, pas, flask, mag, treat = g[0], g[1], g[2], g[3], g[4] if g[4] else 'Control'
            elif pattern == RE_P1_SHORT:
                cl, pas, mag, treat = g[0], g[1], g[2], g[3]; flask = None
            elif pattern in (RE_P1_INV, RE_P1_MAG_FIRST, RE_P1_MINIMAL):
                cl, pas, mag, flask = g[0], g[1], g[2], g[3]; treat = 'Control'
            return {'cell_line_raw': cl, 'passage': int(pas),
                    'flask_size': int(flask) if flask else None, 'magnification': int(mag),
                    'treatment_raw': str(treat).strip(), 'o2_percent': None,
                    'seeding_density': None, 'microscope': None}
        return None

    elif phase == 'PhaseII':
        m = RE_P2_TREAT.match(stem)
        if m: return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                      'passage': int(m.group(3)), 'seeding_density': m.group(4),
                      'flask_size': int(m.group(5)), 'magnification': int(m.group(6)),
                      'o2_percent': None, 'microscope': None}
        m = RE_P2_NOTREAT.match(stem)
        if m: return {'cell_line_raw': m.group(1), 'treatment_raw': 'Control',
                      'passage': int(m.group(2)), 'seeding_density': m.group(3),
                      'flask_size': int(m.group(4)), 'magnification': int(m.group(5)),
                      'o2_percent': None, 'microscope': None}
        m = RE_P2_EXTRA.match(stem)
        if m: return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                      'passage': int(m.group(3)), 'seeding_density': m.group(4),
                      'flask_size': None, 'magnification': int(m.group(5)),
                      'o2_percent': None, 'microscope': None, 'parse_note': 'extra_fields'}
        m = RE_P2_EXTRA_AFTER.match(stem)
        if m: return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                      'passage': int(m.group(3)), 'seeding_density': m.group(4),
                      'flask_size': None, 'magnification': int(m.group(5)),
                      'o2_percent': None, 'microscope': None, 'parse_note': 'extra_after_mag'}
        m = RE_P2_NO_FLASK.match(stem)
        if m: return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                      'passage': int(m.group(3)), 'seeding_density': m.group(4),
                      'flask_size': None, 'magnification': int(m.group(5)),
                      'o2_percent': None, 'microscope': None}
        m = RE_P2_UNKNOWN_DEN.match(stem)
        if m: return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                      'passage': int(m.group(3)), 'seeding_density': 'unknown',
                      'flask_size': int(m.group(4)), 'magnification': int(m.group(5)),
                      'o2_percent': None, 'microscope': None}
        m = RE_P2_TREAT_NOSP.match(stem)
        if m: return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                      'passage': int(m.group(3)), 'seeding_density': m.group(4),
                      'flask_size': int(m.group(5)), 'magnification': int(m.group(6)),
                      'o2_percent': None, 'microscope': None}
        m = RE_P2_DEN_PAREN.match(stem)
        if m: return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                      'passage': int(m.group(3)), 'seeding_density': m.group(4),
                      'flask_size': None, 'magnification': int(m.group(5)),
                      'o2_percent': None, 'microscope': None}
        m = RE_P2_SUFFIX_NC.match(stem)
        if m: return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                      'passage': int(m.group(3)), 'seeding_density': m.group(4),
                      'flask_size': None, 'magnification': int(m.group(5)),
                      'o2_percent': None, 'microscope': None, 'parse_note': 'suffix_no_comma'}
        return None

    elif phase == 'PhaseIII':
        m = RE_P3.match(stem)
        if m: return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                      'passage': int(m.group(3)), 'seeding_density': m.group(4),
                      'flask_size': None, 'magnification': int(m.group(6)),
                      'o2_percent': int(m.group(5)), 'microscope': None}
        m = RE_P3_TPOINT.match(stem)
        if m: return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                      'passage': int(m.group(3)), 'seeding_density': m.group(4),
                      'flask_size': None, 'magnification': int(m.group(6)),
                      'o2_percent': int(m.group(5)), 'microscope': None}
        m = RE_P3_NOO2.match(stem)
        if m: return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                      'passage': int(m.group(3)), 'seeding_density': m.group(4),
                      'flask_size': None, 'magnification': int(m.group(5)),
                      'o2_percent': None, 'microscope': None}
        return None

    elif phase == 'PhaseV':
        stem_clean = re.sub(r'^2(\d{4}-\d{2}-\d{2}_)', r'\1', stem)
        for pattern in [RE_P5, RE_P5_DUP, RE_P5_SUFFIX, RE_P5_DASH, RE_P5_DECIMAL]:
            m = pattern.match(stem_clean)
            if not m: continue
            if pattern == RE_P5_DECIMAL:
                return _p5_result(m, mag_override=int(m.group(6) + m.group(7)))
            return _p5_result(m)
        return None

    return None

# ── Tests v5 ──────────────────────────────────────────────────────────────────
tests = [
    ('PhaseI',  'hFB1-m P10  t175 4 days growth 10x, Ctrl'),
    ('PhaseI',  'hFB2-f P11  t175 5 days growth 20x, a-keto'),
    ('PhaseI',  'hFB2-f P10 3 days growth 10x control'),
    ('PhaseI',  'hFB2-f P17 4 days growth T75 10x control'),
    ('PhaseI',  'hFB2-f P7 10x T175 4 days growth'),
    ('PhaseI',  'hFB1-m P5 10x T75'),
    ('PhaseI',  'hFB1-m P4 10x 4 days growth T75'),
    ('PhaseII', 'hFB12, ctrl, P6, 500kcells, t175, 5-days growth, 10x'),
    ('PhaseII', 'hFB10, P10, 1Mcells, t175, 3-days growth, 10x'),
    ('PhaseII', 'hFB14, Modulators+DEX, P10, 500kcells, t175, 5-days growth, 20x'),
    ('PhaseII', 'hFB14, CTRL, P31, 500kcells, 21-days growth, senescence, 20x'),
    ('PhaseII', 'hFB14, Oligo, P31, 600kcells, 21-days growth, senescence, 10x, fungi infection'),
    ('PhaseII', 'hFB13, CTRL, P33, 1Mcells, 7-days growth, 20x'),
    ('PhaseII', 'hFB13, Mod+DEX, P33, 1-5Mcells, 7-days growth, 10x'),
    ('PhaseII', 'hFB14, CTRL, P32, 500kcells, 21-days growth, 10x, senscence'),
    ('PhaseII', 'hFB8, ctrl, P31, 2-5Mcells, 6-days growth, 10x, post-treatment 2'),
    ('PhaseII', 'hFB8, ctrl, P15, 500kcells, t175, 7-days growth, 20x #2'),
    ('PhaseII', 'hFB12, Oligo, P9, 860kcells, t175, 5-days growth, 10x #2'),
    ('PhaseII', 'hFB12, Mod+DEX, P11, 1,5Mcells, t175, 6-days growth, 20x'),
    ('PhaseII', 'hFB12, Mod, P22, 2-5Mcells, 5-days growth, 20x sparse region'),
    ('PhaseII', 'hFB13, Oligo, P30, unknown, T75, 7-days growth, 10x'),
    ('PhaseII', 'hFB12, DEX P10, 1Mcells, t175, 5-days growth, 20x'),
    ('PhaseII', 'hFB14, CTRL, P15, 1,5Mcells, t175, 7-days growth, 10x copy'),
    ('PhaseII', 'hFB12, CTRL, P38, 1Mcells(Unknown), 10-days growth, 10x'),
    ('PhaseIII','hFB13, DEX, P10, 500kcells, 7-days growth, 3%, 10x'),
    ('PhaseIII','hFB12, ctrl, P11, 1Mcells, 7-days growth, 3%, 10x'),
    ('PhaseIII','hFB12, Contact Inhibtion, P6, 100kcells, 8-days growth, 3%, T0, 10x'),
    ('PhaseIII','hFB11, ctrl, P7, 500kcells, 3-days growth, 10x, post-thaw'),
    ('PhaseV',  '2021-02-24_ASM-01_hFB8_p35_3Ox_20x'),
    ('PhaseV',  '2020-10-21_ASM-01_hFB12_p5_21Ox_10x-2'),
    ('PhaseV',  '2021-03-24_ASM-01_hFB12_p29_3Ox_20x_accumulation'),
    ('PhaseV',  '2020-11-18_ASM-01_hFB8_p19_21Ox_2.0x'),
    ('PhaseV',  '2021-03-31_ASM-01_hFB12_p29_3Ox_10x2_accumulations'),
    ('PhaseV',  '22021-04-07_ASM-01_hFB12_p30_21Ox_10x'),
    ('PhaseV',  '2021-04-07_ASM-01_hFB8_p38_3Ox_10x-acc2'),
]
print('── Tests de parsers v5 ──')
passed = 0
for phase, stem in tests:
    r = parse_image_filename(stem, phase)
    if r:
        cl = normalize_cell_line(r['cell_line_raw'])
        tr = normalize_treatment(r['treatment_raw'])
        note = f'  [{r["parse_note"]}]' if r.get('parse_note') else ''
        print(f'  ✅ [{phase}] cell={cl}, pass={r["passage"]}, treat={tr}, mag={r["magnification"]}x{note}')
        passed += 1
    else:
        print(f'  ❌ [{phase}] FAIL: "{stem}"')
print(f'\n{passed}/{len(tests)} tests pasados')

── Tests de parsers v5 ──
  ✅ [PhaseI] cell=hFB1, pass=10, treat=Control, mag=10x
  ✅ [PhaseI] cell=hFB2, pass=11, treat=a-ketogluturate, mag=20x
  ✅ [PhaseI] cell=hFB2, pass=10, treat=Control, mag=10x
  ✅ [PhaseI] cell=hFB2, pass=17, treat=Control, mag=10x
  ✅ [PhaseI] cell=hFB2, pass=7, treat=Control, mag=10x
  ✅ [PhaseI] cell=hFB1, pass=5, treat=Control, mag=10x
  ✅ [PhaseI] cell=hFB1, pass=4, treat=Control, mag=10x
  ✅ [PhaseII] cell=hFB12, pass=6, treat=Control, mag=10x
  ✅ [PhaseII] cell=hFB10, pass=10, treat=Control, mag=10x
  ✅ [PhaseII] cell=hFB14, pass=10, treat=Oligomycin+DEX, mag=20x
  ✅ [PhaseII] cell=hFB14, pass=31, treat=Control, mag=20x  [extra_fields]
  ✅ [PhaseII] cell=hFB14, pass=31, treat=Oligomycin, mag=10x  [extra_fields]
  ✅ [PhaseII] cell=hFB13, pass=33, treat=Control, mag=20x
  ✅ [PhaseII] cell=hFB13, pass=33, treat=Oligomycin+DEX, mag=10x
  ✅ [PhaseII] cell=hFB14, pass=32, treat=Control, mag=10x  [extra_after_mag]
  ✅ [PhaseII] cell=hFB8, pass=31, treat=Contro

## 5. Escaneo de imágenes

In [17]:
# ══ SECCIÓN 5 — Escaneo de imágenes ═════════════════════════════════════════

records = []
failed  = []
excluded = 0

VALID_EXTENSIONS = {'.jpg', '.jpeg', '.tif', '.tiff', '.png'}

def get_phase(path: Path) -> str | None:
    for part in path.parts:
        if re.match(r'Phase[IVX]+', part, re.IGNORECASE):
            p = part.upper()
            if 'PHASEV' in p and 'PHASEVI' not in p: return 'PhaseV'
            if 'PHASEIII' in p: return 'PhaseIII'
            if 'PHASEII' in p:  return 'PhaseII'
            if 'PHASEI' in p:   return 'PhaseI'
    return None

print('Escaneando imágenes...')
for img_path in IMAGES_ROOT.rglob('*'):
    if img_path.suffix.lower() not in VALID_EXTENSIONS:
        continue
    if should_exclude(img_path):
        excluded += 1
        continue
    phase = get_phase(img_path)
    if phase is None:
        failed.append({'path': str(img_path), 'reason': 'no_phase_detected'})
        continue
    stem = img_path.stem
    parsed = parse_image_filename(stem, phase)
    if parsed is None:
        failed.append({'path': str(img_path), 'phase': phase, 'stem': stem, 'reason': 'parse_fail'})
        continue
    cell_line = normalize_cell_line(parsed['cell_line_raw'])
    sex       = extract_sex_suffix(parsed['cell_line_raw'])
    treatment = normalize_treatment(parsed['treatment_raw'])
    records.append({
        'img_path':        str(img_path),
        'phase':           phase,
        'cell_line':       cell_line,
        'cell_line_sex':   sex,
        'passage':         parsed['passage'],
        'treatment':       treatment,
        'treatment_raw':   parsed['treatment_raw'],
        'magnification':   parsed['magnification'],
        'flask_size':      parsed.get('flask_size'),
        'o2_percent':      parsed.get('o2_percent'),
        'seeding_density': parsed.get('seeding_density'),
        'microscope':      parsed.get('microscope'),
        'image_date':      parsed.get('image_date'),
        'extension':       img_path.suffix.lower(),
    })

df_imgs = pd.DataFrame(records)
df_failed = pd.DataFrame(failed)

print(f'\nImágenes parseadas:  {len(df_imgs):,}')
print(f'Imágenes excluidas:  {excluded:,}')
print(f'Imágenes fallidas:   {len(df_failed):,}')
if len(df_imgs) > 0:
    print('\nDistribución por fase:')
    print(df_imgs.groupby('phase').size())
    print('\nCell lines encontradas:')
    print(sorted(df_imgs['cell_line'].dropna().unique()))

Escaneando imágenes...

Imágenes parseadas:  2,433
Imágenes excluidas:  832
Imágenes fallidas:   2

Distribución por fase:
phase
PhaseI       425
PhaseII     1217
PhaseIII     162
PhaseV       629
dtype: int64

Cell lines encontradas:
['hFB1', 'hFB10', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2', 'hFB5', 'hFB6', 'hFB7', 'hFB8', 'hFB9']


## 5b. Diagnóstico de fallas del parser

In [18]:
# ══ SECCIÓN 5b — Diagnóstico de fallas del parser ═══════════════════════════

if len(df_failed) > 0:
    print(f'=== {len(df_failed)} imágenes fallidas ===')
    print('\nDistribución de razones:')
    print(df_failed['reason'].value_counts())
    for phase in df_failed['phase'].dropna().unique() if 'phase' in df_failed.columns else []:
        subset = df_failed[df_failed['phase'] == phase]
        print(f'\n[{phase}] — {len(subset)} fallas. Primeros 5 stems:')
        for _, row in subset.head(5).iterrows():
            print(f'  "{row.get("stem", "")}"')
    no_phase = df_failed[df_failed['reason'] == 'no_phase_detected']
    if len(no_phase) > 0:
        print(f'\nSin fase detectada ({len(no_phase)} files):')
        for _, row in no_phase.head(5).iterrows():
            print(f'  {row["path"]}')
else:
    print('✅ 0 fallas — todos los archivos parseados correctamente')

=== 2 imágenes fallidas ===

Distribución de razones:
reason
parse_fail    2
Name: count, dtype: int64

[PhaseII] — 2 fallas. Primeros 5 stems:
  "hFB14, CTRL, P15, 1,5Mcells, t175, 7-days growth, 20x copy"
  "hFB14, CTRL, P15, 1,5Mcells, t175, 7-days growth, 10x copy"


## 6. Join imágenes ↔ CSV central



In [19]:
# ══ SECCIÓN 6 — Join imágenes ↔ CSV central ════════════════════════════════

# ── Tabla de normalización EXTENDIDA para imágenes ────────────────────────────
# Todos los valores deben mapear al mismo token que usa el CSV de Sturm
TREATMENT_MAP_EXTENDED = {
    'ctrl': 'Control', 'control': 'Control', 'contorl': 'Control',
    '(1m), control': 'Control', '(2m), control': 'Control',
    '(700k) - control': 'Control', '': 'Control',
    'dex': 'DEX',
    '(1m), dex': 'DEX', '(2m), dex': 'DEX', '(2m),  dex': 'DEX',
    '(2m) - dex': 'DEX', 'dex vf': 'DEX',
    'puls': 'Pulsated_DEX', 'dex-puls': 'Pulsated_DEX',
    '(1m), puls': 'Pulsated_DEX', '(2m), puls': 'Pulsated_DEX',
    '(2m),  puls': 'Pulsated_DEX', '(500k) - puls': 'Pulsated_DEX',
    '(500k), puls': 'Pulsated_DEX',
    'mitoq': 'MitoQ',
    '(1m), mitoq': 'MitoQ', '(1m) - mitoq': 'MitoQ', '(2m), mitoq': 'MitoQ',
    'mitoq+dex': 'MitoQ+DEX', 'mitoq + dex': 'MitoQ+DEX',
    '(1m), mitoq + dex': 'MitoQ+DEX', '(2m), mitoq + dex': 'MitoQ+DEX',
    '(2m) - mitoq + dex': 'MitoQ+DEX', '(2m), mq + dex': 'MitoQ+DEX',
    'nac': 'NAC',
    '(1m), nac': 'NAC', '(2m), nac': 'NAC', '(750k) - nac': 'NAC',
    'nac+dex': 'NAC+DEX', 'nac + dex': 'NAC+DEX', 'nac +dex': 'NAC+DEX',
    '(1m), nac + dex': 'NAC+DEX', '(2m), nac + dex': 'NAC+DEX',
    '(2m) - nac + dex': 'NAC+DEX',
    # AKG → valor del CSV de Sturm (typo intencional: 'a-ketogluturate' con u)
    'akg': 'a-ketogluturate', 'a-keto': 'a-ketogluturate',
    'a-ketoglutarate': 'a-ketogluturate',
    'a-ketogluturate': 'a-ketogluturate',
    '(1m), a-keto': 'a-ketogluturate', '(1m) - a-keto': 'a-ketogluturate',
    '(2m), a-keto': 'a-ketogluturate',
    'oligomycin': 'Oligomycin', 'oligo': 'Oligomycin',
    'oligomycin+dex': 'Oligomycin+DEX', 'oligo+dex': 'Oligomycin+DEX',
    'modulators': 'Oligomycin', 'mod': 'Oligomycin',
    'modulators+dex': 'Oligomycin+DEX', 'mod+dex': 'Oligomycin+DEX',
    'mod-dex': 'Oligomycin+DEX',
    'contactinhibition': 'Contact_Inhibition',
    'contact inhibition': 'Contact_Inhibition',
    'contact inhibtion': 'Contact_Inhibition',
    'post-thaw': 'Control', 'arrival': 'Control', 'vehicle': 'Control',
}

def normalize_treatment_v2(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return 'Control'
    key = str(raw).strip().lower()
    return TREATMENT_MAP_EXTENDED.get(key, str(raw).strip())

# ── Preparar df_imgs_join ─────────────────────────────────────────────────────
PHASE_TO_STUDY_PART = {'PhaseI': 1, 'PhaseII': 2, 'PhaseIII': 3, 'PhaseV': 5}

df_imgs_join = df_imgs.copy()
df_imgs_join['passage']        = pd.to_numeric(df_imgs_join['passage'], errors='coerce').astype('Int64')
df_imgs_join['study_part']     = df_imgs_join['phase'].map(PHASE_TO_STUDY_PART).astype('Int64')
df_imgs_join['percent_oxygen'] = df_imgs_join['o2_percent'].fillna(21).astype('Int64')
df_imgs_join['treatment_norm'] = df_imgs_join['treatment'].apply(normalize_treatment_v2)
df_imgs_join['time_point']     = pd.NA

unmapped = df_imgs_join[
    ~df_imgs_join['treatment'].isna() &
    (df_imgs_join['treatment_norm'] == df_imgs_join['treatment'])
]['treatment'].value_counts()
unmapped = unmapped[~unmapped.index.str.lower().isin(TREATMENT_MAP_EXTENDED.keys())]
if len(unmapped) > 0:
    print(f'⚠️  Treatments sin mapear ({len(unmapped)} valores únicos):')
    print(unmapped.head(15))
else:
    print('✅ Todos los treatments de imágenes mapeados')

# ── Preparar df_csv_join ──────────────────────────────────────────────────────
csv_keep_cols = ['sample_id', 'cell_line', 'passage', 'treatment_norm', 'study_part',
                 'percent_oxygen', 'time_point', 'pdl', 'pdl_norm', 'pdl_bin',
                 'percent_lifespan', 'clinical_condition', 'donor_age', 'cell_type',
                 'days_grown', 'telomere_length', 'mtdna_cn', 'cell_size', 'cell_volume',
                 'seahorse_respiration', 'seahorse_ecar', 'rnaseq_id', 'methylation_basename'] +                 [c for c in df.columns if c.startswith('clock_')]
csv_keep_cols = [c for c in csv_keep_cols if c in df.columns]

df_csv_join = df[csv_keep_cols].copy()
df_csv_join['passage']        = pd.to_numeric(df_csv_join['passage'], errors='coerce').astype('Int64')
df_csv_join['study_part']     = df_csv_join['study_part'].astype('Int64')
df_csv_join['percent_oxygen'] = df_csv_join['percent_oxygen'].astype('Int64')
df_csv_join['time_point']     = df_csv_join['time_point'].astype('Int64')

# ── Limpiar NaN en llaves ANTES de separar CI ─────────────────────────────────
n_before = len(df_csv_join)
df_csv_join = df_csv_join.dropna(subset=['cell_line', 'passage']).copy()
print(f'CSV: {n_before} → {len(df_csv_join)} ({n_before - len(df_csv_join)} eliminadas por NaN)')

n_before = len(df_imgs_join)
df_imgs_join = df_imgs_join.dropna(subset=['cell_line', 'passage']).copy()
print(f'Imgs: {n_before} → {len(df_imgs_join)} ({n_before - len(df_imgs_join)} eliminadas por NaN)')

# ── Validación de unicidad AQUÍ (después del dropna) ─────────────────────────
JOIN_KEYS_BASE = ['cell_line', 'passage', 'treatment_norm', 'study_part', 'percent_oxygen']
key_cols = JOIN_KEYS_BASE
dup_check = df_csv_join.groupby(key_cols, dropna=False).size()
n_dups = (dup_check > 1).sum()
if n_dups > 0:
    dup_examples = dup_check[dup_check > 1].reset_index()
    non_ci = dup_examples[~dup_examples['treatment_norm'].str.contains('Contact|Inhibit', na=False)]
    if len(non_ci) > 0:
        print(f'⛔ {len(non_ci)} quíntuplas duplicadas fuera de CI — revisar datos')
        print(non_ci.head(5))
    else:
        print(f'✅ Unicidad OK — {len(dup_examples)} duplicados son Contact Inhibition (esperado)')
else:
    print('✅ Unicidad perfecta — todas las quíntuplas son únicas')

# ── Separar Contact Inhibition ────────────────────────────────────────────────
is_ci_img = df_imgs_join['treatment_norm'].str.contains('Contact_Inhibition', na=False)
is_ci_csv = df_csv_join['treatment_norm'].str.contains('Contact_Inhibition', na=False)

df_imgs_standard = df_imgs_join[~is_ci_img].copy()
df_imgs_ci       = df_imgs_join[is_ci_img].copy()
df_csv_standard  = df_csv_join[~is_ci_csv].copy()
df_csv_ci        = df_csv_join[is_ci_csv].copy()

print(f'\nEstándar: {len(df_imgs_standard):,} imgs / {len(df_csv_standard):,} CSV rows')
print(f'CI:       {len(df_imgs_ci):,} imgs / {len(df_csv_ci):,} CSV rows')

dups_std = df_csv_standard.duplicated(subset=JOIN_KEYS_BASE, keep=False)
if dups_std.sum() > 0:
    print(f'⛔ {dups_std.sum()} duplicados en CSV estándar')
    raise ValueError('Resolver duplicados antes de continuar')
else:
    print('✅ CSV estándar: quíntupla única en todas las filas')

# ── JOIN estándar ─────────────────────────────────────────────────────────────
df_joined_standard = df_imgs_standard.merge(df_csv_standard, on=JOIN_KEYS_BASE,
                                              how='left', validate='many_to_one')

# ── JOIN Contact Inhibition (T0 como representativo) ─────────────────────────
df_csv_ci_t0 = df_csv_ci.sort_values('time_point').drop_duplicates(
    subset=JOIN_KEYS_BASE, keep='first').copy()
df_joined_ci = df_imgs_ci.merge(df_csv_ci_t0, on=JOIN_KEYS_BASE,
                                 how='left', validate='many_to_one')

df_manifest = pd.concat([df_joined_standard, df_joined_ci], ignore_index=True)

# ── Validaciones ──────────────────────────────────────────────────────────────
print('\n' + '═'*55)
print('VALIDACIONES DE INTEGRIDAD')
print('═'*55)
n_imgs_total = len(df_imgs_join)
n_manifest   = len(df_manifest)
check1 = n_imgs_total == n_manifest
print(f'{"✅" if check1 else "⛔"} Filas conservadas: {n_manifest:,} (esperado {n_imgs_total:,})')
check2 = df_manifest.duplicated(subset=['img_path']).sum() == 0
print(f'{"✅" if check2 else "⛔"} img_path único: {df_manifest["img_path"].nunique():,} únicos')
n_matched = df_manifest['pdl'].notna().sum()
pct = 100 * n_matched / n_manifest
check3 = pct >= 85
print(f'{"✅" if check3 else "⚠️ "} Match PDL: {n_matched:,} / {n_manifest:,} ({pct:.1f}%)')
VALID_CELL_LINES = {'hFB1','hFB2','hFB6','hFB7','hFB8','hFB11','hFB12','hFB13','hFB14'}
unexpected = set(df_manifest['cell_line'].dropna().unique()) - VALID_CELL_LINES
check4 = len(unexpected) == 0
print(f'{"✅" if check4 else "⚠️ "} Cell lines válidas: {unexpected if unexpected else "todas OK"}')
print('═'*55)

df_manifest['has_image']       = True
df_manifest['has_pdl']         = df_manifest['pdl'].notna()
df_manifest['has_rna']         = df_manifest['rnaseq_id'].notna()          if 'rnaseq_id'            in df_manifest.columns else False
df_manifest['has_methylation'] = df_manifest['methylation_basename'].notna() if 'methylation_basename' in df_manifest.columns else False
df_manifest['has_telomere']    = df_manifest['telomere_length'].notna()    if 'telomere_length'      in df_manifest.columns else False
df_manifest['has_mtdna']       = df_manifest['mtdna_cn'].notna()           if 'mtdna_cn'             in df_manifest.columns else False
df_manifest['n_modalities']    = df_manifest[['has_rna','has_methylation','has_telomere','has_mtdna']].astype(int).sum(axis=1)

print('\nModalidades en imágenes con PDL:')
with_pdl = df_manifest[df_manifest['has_pdl']]
for col in ['has_rna','has_methylation','has_telomere','has_mtdna']:
    n = with_pdl[col].sum()
    print(f'  {col:22s}: {n:4,} / {len(with_pdl):,}')

⚠️  Treatments sin mapear (2 valores únicos):
treatment
Pulsated_DEX          30
Contact_Inhibition    12
Name: count, dtype: int64
CSV: 1919 → 1901 (18 eliminadas por NaN)
Imgs: 2433 → 2433 (0 eliminadas por NaN)
✅ Unicidad OK — 7 duplicados son Contact Inhibition (esperado)

Estándar: 2,421 imgs / 1,815 CSV rows
CI:       12 imgs / 86 CSV rows
✅ CSV estándar: quíntupla única en todas las filas

═══════════════════════════════════════════════════════
VALIDACIONES DE INTEGRIDAD
═══════════════════════════════════════════════════════
✅ Filas conservadas: 2,433 (esperado 2,433)
✅ img_path único: 2,433 únicos
✅ Match PDL: 2,261 / 2,433 (92.9%)
⚠️  Cell lines válidas: {'hFB10', 'hFB5', 'hFB9'}
═══════════════════════════════════════════════════════

Modalidades en imágenes con PDL:
  has_rna               :  366 / 2,261
  has_methylation       :  564 / 2,261
  has_telomere          :  552 / 2,261
  has_mtdna             :  562 / 2,261


## 6b. Diagnóstico de imágenes sin match (sin PDL)

In [20]:
# ══ SECCIÓN 6b — Clasificación formal de imágenes sin match ════════════════

unmatched = df_manifest[~df_manifest['has_pdl']].copy()

if len(unmatched) > 0:
    def classify_unmatched(row):
        if row['cell_line'] in {'hFB5', 'hFB9', 'hFB10'}:
            return 'cell_line_not_in_csv'
        if row['study_part'] == 5:
            return 'phaseV_late_passage_no_csv_entry'
        if row['study_part'] == 2 and row['treatment_norm'] in {'Oligomycin+DEX', 'Oligomycin'}:
            return 'phaseII_treatment_discontinued'
        return 'other'

    unmatched['unmatched_reason'] = unmatched.apply(classify_unmatched, axis=1)
    print(f'=== {len(unmatched)} imágenes sin match — clasificación ===')
    print(unmatched['unmatched_reason'].value_counts())
    print(f'\nEstas imágenes NO tienen error — son lagunas reales del dataset.')
    print(f'Se excluyen del entrenamiento (has_pdl=False) pero se conservan en el manifest.')

    df_manifest = df_manifest.merge(
        unmatched[['img_path','unmatched_reason']], on='img_path', how='left')
    df_manifest['unmatched_reason'] = df_manifest['unmatched_reason'].fillna('matched')
else:
    print('✅ Todas las imágenes tienen match')

=== 172 imágenes sin match — clasificación ===
unmatched_reason
phaseII_treatment_discontinued      132
phaseV_late_passage_no_csv_entry     28
other                                 6
cell_line_not_in_csv                  6
Name: count, dtype: int64

Estas imágenes NO tienen error — son lagunas reales del dataset.
Se excluyen del entrenamiento (has_pdl=False) pero se conservan en el manifest.


## 7. Integrar metadata de GEO (RNA-seq y Metilación)

In [21]:
# ══ SECCIÓN 7 — Integrar metadata de GEO (CORREGIDA) ═══════════════════════
#
# FIX: La versión anterior usaba char_study_sample_id como join key para ambos,
# pero:
#   - rnaseq_id del manifest (rango 1-252) ≠ char_study_sample_id (rango 318-1907)
#   - methylation_basename del manifest (texto "203784950028_R04C01") ≠ char_study_sample_id (entero)
#
# CORRECCIÓN:
#   - RNA:  unir rnaseq_id ↔ char_rnaseq_sampleid (ambos rango 1-360)
#   - MET:  construir sentrix_basename = char_slide + "_" + char_array en GEO,
#           luego unir methylation_basename ↔ sentrix_basename
#
# IMPACTO: Los flags has_rna/has_methylation NO cambian (se calculan antes).
#          Solo se llenan las columnas rna_char_*/meth_char_* que antes estaban vacías.
# ════════════════════════════════════════════════════════════════════════════════

df_rna_meta  = pd.read_csv(GEO_RNA_META)  if GEO_RNA_META.exists()  else None
df_meth_meta = pd.read_csv(GEO_METH_META) if GEO_METH_META.exists() else None

def normalize_join_key(series: pd.Series) -> pd.Series:
    """Normaliza una serie a string limpio para joins robustos."""
    s = series.astype("string").str.strip()
    s = s.replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "<NA>": pd.NA})
    s = s.str.replace(r"^(-?\d+)\.0$", r"\1", regex=True)
    s = s.str.lower()
    return s

# Limpiar columnas GEO previas (por si se re-ejecuta la celda)
cols_to_drop = [c for c in df_manifest.columns if c.startswith("rna_") or c.startswith("meth_")]
if cols_to_drop:
    df_manifest = df_manifest.drop(columns=cols_to_drop)
    print(f"🧹 Limpiadas {len(cols_to_drop)} columnas GEO previas")

n_before = len(df_manifest)

# ── RNA-seq: unir rnaseq_id ↔ char_rnaseq_sampleid ───────────────────────────
if df_rna_meta is not None and "rnaseq_id" in df_manifest.columns:
    RNA_JOIN_KEY_GEO = "char_rnaseq_sampleid"

    if RNA_JOIN_KEY_GEO not in df_rna_meta.columns:
        print(f"⚠️  Columna '{RNA_JOIN_KEY_GEO}' no encontrada en GEO RNA metadata")
        print(f"   Columnas disponibles: {df_rna_meta.columns.tolist()}")
    else:
        # Renombrar todas las columnas GEO con prefijo rna_
        rna_cols_rename = {c: f"rna_{c}" for c in df_rna_meta.columns if c != RNA_JOIN_KEY_GEO}
        df_rna_join = df_rna_meta.rename(columns=rna_cols_rename).copy()

        # Preparar keys normalizadas
        df_manifest = df_manifest.copy()
        df_manifest["_rna_join_key"] = normalize_join_key(df_manifest["rnaseq_id"])
        df_rna_join["_rna_join_key"] = normalize_join_key(df_rna_join[RNA_JOIN_KEY_GEO])

        # Diagnóstico pre-join
        manifest_keys = set(df_manifest["_rna_join_key"].dropna())
        geo_keys = set(df_rna_join["_rna_join_key"].dropna())
        overlap = len(manifest_keys & geo_keys)
        print(f"🔎 RNA join: manifest rnaseq_id ({len(manifest_keys)} unique) "
              f"↔ GEO {RNA_JOIN_KEY_GEO} ({len(geo_keys)} unique) → overlap={overlap}")

        # Deduplicar GEO por key (conservar primera)
        dups = df_rna_join["_rna_join_key"].duplicated(keep=False).sum()
        if dups > 0:
            print(f"   ⚠️ {dups} filas duplicadas en GEO RNA; conservando primera por key")
            df_rna_join = df_rna_join.drop_duplicates("_rna_join_key", keep="first")

        # Join
        df_manifest = df_manifest.merge(
            df_rna_join.drop(columns=[RNA_JOIN_KEY_GEO]),
            on="_rna_join_key",
            how="left"
        ).drop(columns=["_rna_join_key"])

        assert len(df_manifest) == n_before, (
            f"⛔ Join RNA expandió filas: {n_before} → {len(df_manifest)}"
        )

        # Verificar resultado
        rna_geo_cols = [c for c in df_manifest.columns if c.startswith("rna_")]
        n_filled = int(df_manifest[rna_geo_cols].notna().any(axis=1).sum()) if rna_geo_cols else 0
        print(f"✅ RNA GEO integrado: {n_filled} filas con metadata ({len(rna_geo_cols)} columnas)")

# ── Metilación: construir sentrix_basename, unir ↔ methylation_basename ───────
if df_meth_meta is not None and "methylation_basename" in df_manifest.columns:

    # Construir la key de join: sentrix_basename = slide + "_" + array
    if "char_slide" in df_meth_meta.columns and "char_array" in df_meth_meta.columns:
        df_meth_meta = df_meth_meta.copy()
        df_meth_meta["sentrix_basename"] = (
            df_meth_meta["char_slide"].astype(str).str.strip()
            + "_"
            + df_meth_meta["char_array"].astype(str).str.strip()
        )
        METH_JOIN_KEY_GEO = "sentrix_basename"
        print(f"🔧 Construido sentrix_basename en GEO met "
              f"({df_meth_meta['sentrix_basename'].nunique()} únicos)")
    else:
        print("⚠️  Columnas char_slide/char_array no encontradas en GEO Met metadata")
        print(f"   Columnas disponibles: {df_meth_meta.columns.tolist()}")
        METH_JOIN_KEY_GEO = None

    if METH_JOIN_KEY_GEO is not None:
        # Renombrar columnas GEO con prefijo meth_
        meth_cols_rename = {c: f"meth_{c}" for c in df_meth_meta.columns if c != METH_JOIN_KEY_GEO}
        df_meth_join = df_meth_meta.rename(columns=meth_cols_rename).copy()

        # Preparar keys normalizadas
        df_manifest = df_manifest.copy()
        df_manifest["_meth_join_key"] = normalize_join_key(df_manifest["methylation_basename"])
        df_meth_join["_meth_join_key"] = normalize_join_key(df_meth_join[METH_JOIN_KEY_GEO])

        # Diagnóstico pre-join
        manifest_keys_met = set(df_manifest["_meth_join_key"].dropna())
        geo_keys_met = set(df_meth_join["_meth_join_key"].dropna())
        overlap_met = len(manifest_keys_met & geo_keys_met)
        print(f"🔎 Met join: manifest methylation_basename ({len(manifest_keys_met)} unique) "
              f"↔ GEO sentrix_basename ({len(geo_keys_met)} unique) → overlap={overlap_met}")

        # Deduplicar GEO por key
        dups_met = df_meth_join["_meth_join_key"].duplicated(keep=False).sum()
        if dups_met > 0:
            print(f"   ⚠️ {dups_met} filas duplicadas en GEO Met; conservando primera por key")
            df_meth_join = df_meth_join.drop_duplicates("_meth_join_key", keep="first")

        # Join
        df_manifest = df_manifest.merge(
            df_meth_join.drop(columns=[METH_JOIN_KEY_GEO]),
            on="_meth_join_key",
            how="left"
        ).drop(columns=["_meth_join_key"])

        assert len(df_manifest) == n_before, (
            f"⛔ Join Met expandió filas: {n_before} → {len(df_manifest)}"
        )

        # Verificar resultado
        meth_geo_cols = [c for c in df_manifest.columns if c.startswith("meth_")]
        n_filled_met = int(df_manifest[meth_geo_cols].notna().any(axis=1).sum()) if meth_geo_cols else 0
        print(f"✅ Met GEO integrado: {n_filled_met} filas con metadata ({len(meth_geo_cols)} columnas)")

print(f"\nManifest post-GEO: {df_manifest.shape}")
print(f"  Filas: {len(df_manifest)} (sin cambio: {'✅' if len(df_manifest) == n_before else '⛔'})")

🔎 RNA join: manifest rnaseq_id (148 unique) ↔ GEO char_rnaseq_sampleid (345 unique) → overlap=148
✅ RNA GEO integrado: 366 filas con metadata (41 columnas)
🔧 Construido sentrix_basename en GEO met (479 únicos)
🔎 Met join: manifest methylation_basename (257 unique) ↔ GEO sentrix_basename (479 unique) → overlap=254
✅ Met GEO integrado: 558 filas con metadata (38 columnas)

Manifest post-GEO: (2433, 130)
  Filas: 2433 (sin cambio: ✅)


## 8. GroupKFold splits por cell_line (anti-leakage)

In [22]:
# ══ SECCIÓN 8 — Splits manuales 3 folds (anti-leakage por cell_line) ════════
#
# Con 9 cell lines activas en FINETUNE, 5 folds dejaría folds con <30 imágenes.
# 3 folds garantiza ≥80 imágenes por fold y rango PDL amplio.
#
# Fold 0: hFB12            — 247 imgs, PDL 2–67, PhaseII+III+V
# Fold 1: hFB13,14,6,8     — PDL 2–80, PhaseII+III+V
# Fold 2: hFB1,2,11,7      — PhaseI+II+III, fases no cubiertas por folds 0/1

FOLD_ASSIGNMENT = {
    'hFB12': 0,
    'hFB13': 1, 'hFB14': 1, 'hFB6': 1, 'hFB8': 1,
    'hFB1':  2, 'hFB2':  2, 'hFB11': 2, 'hFB7': 2,
}

df_manifest['fold'] = df_manifest['cell_line'].map(FOLD_ASSIGNMENT)

assert df_manifest.groupby('cell_line')['fold'].nunique().max() == 1, \
    '⛔ Una cell line aparece en más de un fold'

print('═'*60)
print('SPLITS — 3 FOLDS (anti-leakage por cell_line)')
print('═'*60)

subset = df_manifest[df_manifest['has_pdl']].dropna(subset=['fold'])
summary = subset.groupby('fold').agg(
    n          = ('img_path', 'count'),
    cell_lines = ('cell_line', lambda x: ', '.join(sorted(x.dropna().unique()))),
    phases     = ('phase',     lambda x: ', '.join(sorted(x.dropna().unique()))),
    pdl_min    = ('pdl', 'min'),
    pdl_max    = ('pdl', 'max'),
    pct_early  = ('pdl_bin', lambda x: (x=='early').mean()),
    pct_mid    = ('pdl_bin', lambda x: (x=='mid').mean()),
    pct_late   = ('pdl_bin', lambda x: (x=='late').mean()),
).round(2)
print(summary.to_string())

checks = [
    ('Todos los folds con datos',              summary['n'].min() > 0),
    ('Rango PDL > 20 en todos los folds',      (summary['pdl_max'] - summary['pdl_min']).min() >= 20),
    ('Ningún fold >80% late',                  summary['pct_late'].max() <= 0.80),
    ('Ningún fold <10% early',                 summary['pct_early'].min() >= 0.10),
    ('0 cell lines en dos folds distintos',    df_manifest.groupby('cell_line')['fold'].nunique().max() == 1),
]
print('\n── Checklist splits ──')
for desc, passed in checks:
    print(f'  {"✅" if passed else "❌"} {desc}')

df_manifest['fold_role'] = df_manifest['fold']
print('\n✅ Splits asignados')
print(df_manifest['fold'].value_counts(dropna=False).sort_index())

════════════════════════════════════════════════════════════
SPLITS — 3 FOLDS (anti-leakage por cell_line)
════════════════════════════════════════════════════════════
         n                cell_lines                             phases  pdl_min  pdl_max  pct_early  pct_mid  pct_late
fold                                                                                                                   
0.0    491                     hFB12          PhaseII, PhaseIII, PhaseV     4.06    67.16       0.30     0.37      0.33
1.0   1163  hFB13, hFB14, hFB6, hFB8          PhaseII, PhaseIII, PhaseV     4.39    80.84       0.23     0.37      0.39
2.0    607   hFB1, hFB11, hFB2, hFB7  PhaseI, PhaseII, PhaseIII, PhaseV     2.82    60.36       0.09     0.44      0.47

── Checklist splits ──
  ✅ Todos los folds con datos
  ✅ Rango PDL > 20 en todos los folds
  ✅ Ningún fold >80% late
  ❌ Ningún fold <10% early
  ✅ 0 cell lines en dos folds distintos

✅ Splits asignados
fold
0.0     555
1.0    126

## 9. Export manifest versionado con MD5

In [23]:
# ══ SECCIÓN 9 — Export manifest versionado ══════════════════════════════════

import hashlib, datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

out_full = OUTPUT_DIR / f'manifest_full_{timestamp}.csv'
df_manifest.to_csv(out_full, index=False)
md5 = hashlib.md5(open(out_full,'rb').read()).hexdigest()
print(f'Manifest completo guardado:')
print(f'  {out_full}')
print(f'  Shape: {df_manifest.shape}')
print(f'  MD5:   {md5}')

meta = {
    'timestamp':      timestamp,
    'n_total_images': len(df_manifest),
    'n_with_pdl':     int(df_manifest['has_pdl'].sum()),
    'n_failed_parse': len(df_failed),
    'n_excluded':     excluded,
    'cell_lines':     sorted(df_manifest['cell_line'].dropna().unique().tolist()),
    'phases':         sorted(df_manifest['phase'].dropna().unique().tolist()),
    'pdl_range':      [float(df_manifest['pdl'].min()), float(df_manifest['pdl'].max())],
    'md5':            md5,
    'csv_path':       str(CSV_PATH),
    'images_root':    str(IMAGES_ROOT),
}
meta_path = OUTPUT_DIR / f'manifest_meta_{timestamp}.json'
json.dump(meta, open(meta_path,'w'), indent=2)
print(f'\nMetadata guardada: {meta_path}')

Manifest completo guardado:
  /Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS/manifest_full_20260327_152829.csv
  Shape: (2433, 132)
  MD5:   4ba8c025cf1c74fdc74e7634a39412bb

Metadata guardada: /Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS/manifest_meta_20260327_152829.json


## 10. Subconjunto MVP-1
Para el primer encoder visual: solo imágenes con PDL disponible, solo tratamiento Control, 10x.

Excluyendo SURF1 del baseline (donantes con enfermedad mitocondrial) — se pueden agregar luego para robustez.

In [24]:
# ══ SECCIÓN 10 — Subconjuntos MVP-1 ════════════════════════════════════════
#
# PRETRAIN  → todos los treatments + SURF1 (máx. datos, aprende features morfológicos)
# FINETUNE  → solo Control, sin SURF1 (calibra PDL sin confusión farmacológica)
# EVAL      → Control 10x, sin SURF1 (métrica oficial comparable con ext.)

EXCLUDE_TREATMENTS = {'Contact_Inhibition', 'Contact_Inhibition_Regrowth'}
EXCLUDE_CELL_LINES  = {'hFB5', 'hFB9', 'hFB10'}

mvp1_base = df_manifest[
    df_manifest['has_pdl'] &
    ~df_manifest['treatment_norm'].isin(EXCLUDE_TREATMENTS) &
    ~df_manifest['cell_line'].isin(EXCLUDE_CELL_LINES)
].copy()

is_surf1 = mvp1_base['clinical_condition'].str.lower().str.contains('surf', na=False) \
           if 'clinical_condition' in mvp1_base.columns \
           else pd.Series(False, index=mvp1_base.index)

mvp1_pretrain = mvp1_base.copy()
mvp1_finetune = mvp1_base[~is_surf1 & (mvp1_base['treatment_norm'] == 'Control')].copy()
mvp1_eval     = mvp1_finetune[mvp1_finetune['magnification'] == 10].copy()

print('═'*60)
print('MVP-1 — TRES SUBCONJUNTOS')
print('═'*60)
for label, subset in [
    ('PRETRAIN  (todos treatments + SURF1)', mvp1_pretrain),
    ('FINETUNE  (solo Control, sin SURF1)',  mvp1_finetune),
    ('EVAL      (Control 10x, sin SURF1)',   mvp1_eval),
]:
    print(f'\n── {label} ──')
    print(f'  n imágenes:  {len(subset):,}')
    print(f'  cell lines:  {sorted(subset["cell_line"].dropna().unique())}')
    print(f'  fases:       {sorted(subset["phase"].dropna().unique())}')
    print(f'  PDL range:   {subset["pdl"].min():.1f} – {subset["pdl"].max():.1f}')
    print(f'  PDL bins:    {dict(subset["pdl_bin"].value_counts())}')
    print(f'  magnif:      {dict(subset["magnification"].value_counts().sort_index())}')

print('\n── Splits en FINETUNE ──')
if 'fold' in mvp1_finetune.columns:
    print(mvp1_finetune.groupby('fold').agg(
        n=('img_path','count'),
        cell_lines=('cell_line', lambda x: ', '.join(sorted(x.dropna().unique()))),
        pdl_min=('pdl','min'), pdl_max=('pdl','max'),
    ).round(1).to_string())

print('\n── Checklist MVP-1 ──')
checks = [
    ('PRETRAIN  ≥ 1,500 imágenes',         len(mvp1_pretrain) >= 1500),
    ('FINETUNE  ≥ 300 imágenes',           len(mvp1_finetune) >= 300),
    ('EVAL      ≥ 150 imágenes',           len(mvp1_eval) >= 150),
    ('FINETUNE cubre early/mid/late',      mvp1_finetune['pdl_bin'].nunique() == 3),
    ('EVAL cubre ≥ 4 cell lines',          mvp1_eval['cell_line'].nunique() >= 4),
    ('EVAL cubre todas las fases',         mvp1_eval['phase'].nunique() == 4),
    ('PDL disponible en >95% FINETUNE',    mvp1_finetune['has_pdl'].mean() > 0.95),
    ('Splits asignados en FINETUNE',       'fold' in mvp1_finetune.columns and
                                            mvp1_finetune['fold'].notna().any()),
]
for desc, passed in checks:
    print(f'  {"✅" if passed else "❌"} {desc}')

# Guardar
out_pretrain = OUTPUT_DIR / f'manifest_mvp1_pretrain_{timestamp}.csv'
out_finetune = OUTPUT_DIR / f'manifest_mvp1_finetune_{timestamp}.csv'
out_eval     = OUTPUT_DIR / f'manifest_mvp1_eval_{timestamp}.csv'
mvp1_pretrain.to_csv(out_pretrain, index=False)
mvp1_finetune.to_csv(out_finetune, index=False)
mvp1_eval.to_csv(out_eval, index=False)
for path in [out_pretrain, out_finetune, out_eval]:
    md5_sub = hashlib.md5(open(path,'rb').read()).hexdigest()
    print(f'\n  {path.name}  MD5: {md5_sub}')

════════════════════════════════════════════════════════════
MVP-1 — TRES SUBCONJUNTOS
════════════════════════════════════════════════════════════

── PRETRAIN  (todos treatments + SURF1) ──
  n imágenes:  2,249
  cell lines:  ['hFB1', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2', 'hFB6', 'hFB7', 'hFB8']
  fases:       ['PhaseI', 'PhaseII', 'PhaseIII', 'PhaseV']
  PDL range:   2.8 – 80.8
  PDL bins:    {'late': 908, 'mid': 879, 'early': 462}
  magnif:      {4: 2, 10: 1142, 20: 1105}

── FINETUNE  (solo Control, sin SURF1) ──
  n imágenes:  763
  cell lines:  ['hFB1', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2']
  fases:       ['PhaseI', 'PhaseII', 'PhaseIII', 'PhaseV']
  PDL range:   2.8 – 80.8
  PDL bins:    {'late': 463, 'mid': 155, 'early': 145}
  magnif:      {4: 2, 10: 393, 20: 368}

── EVAL      (Control 10x, sin SURF1) ──
  n imágenes:  393
  cell lines:  ['hFB1', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2']
  fases:       ['PhaseI', 'PhaseII', 'PhaseIII', 'PhaseV']
  PDL range:   

In [25]:
# ══ CONTRATO DEL MANIFEST — falla duro si algo está mal ════════════════════
# Correr DESPUÉS de la sección 10 (necesita mvp1_pretrain y mvp1_finetune).

print('Validando contrato del manifest...')

assert df_manifest['img_path'].nunique() == len(df_manifest), \
    f'⛔ img_path no es único: {len(df_manifest) - df_manifest["img_path"].nunique()} duplicados'

N_IMGS_ESPERADO = len(df_imgs_join)
assert len(df_manifest) == N_IMGS_ESPERADO, \
    f'⛔ Filas incorrectas: esperado {N_IMGS_ESPERADO}, actual {len(df_manifest)}'

VALID_CELL_LINES_CONTRACT = {'hFB1','hFB2','hFB6','hFB7','hFB8','hFB11','hFB12','hFB13','hFB14','hFB5','hFB9','hFB10'}
unexpected_cl = set(df_manifest['cell_line'].dropna().unique()) - VALID_CELL_LINES_CONTRACT
assert not unexpected_cl, f'⛔ Cell lines fuera del dominio: {unexpected_cl}'

pct_pdl = df_manifest['has_pdl'].mean()
assert pct_pdl >= 0.85, f'⛔ Match PDL insuficiente: {pct_pdl:.1%} (mínimo 85%)'

REQUIRED_COLS = ['img_path','cell_line','passage','pdl','pdl_norm','pdl_bin',
                 'phase','treatment_norm','study_part','percent_oxygen','magnification','has_pdl','fold']
missing = [c for c in REQUIRED_COLS if c not in df_manifest.columns]
assert not missing, f'⛔ Columnas requeridas ausentes: {missing}'

pdl_norm_valid = df_manifest['pdl_norm'].dropna()
assert pdl_norm_valid.between(0, 1).all(), \
    f'⛔ pdl_norm fuera de [0,1]: min={pdl_norm_valid.min():.3f} max={pdl_norm_valid.max():.3f}'

cl_fold = df_manifest.dropna(subset=['cell_line','fold']).groupby('cell_line')['fold'].nunique()
assert cl_fold.max() == 1, f'⛔ Cell lines en múltiples folds: {cl_fold[cl_fold>1].index.tolist()}'

# Assert MVP-1 (ahora sí existen las variables)
ft_cls = set(mvp1_finetune['cell_line'].dropna().unique())
pt_cls = set(mvp1_pretrain['cell_line'].dropna().unique())
assert ft_cls.issubset(pt_cls), f'⛔ FINETUNE tiene cell lines no en PRETRAIN: {ft_cls - pt_cls}'

print('═'*55)
print('✅ CONTRATO DEL MANIFEST VERIFICADO')
print('═'*55)
print(f'  Filas:          {len(df_manifest):,}')
print(f'  Match PDL:      {pct_pdl:.1%}')
print(f'  Cell lines:     {sorted(df_manifest["cell_line"].dropna().unique())}')
print(f'  Folds:          {sorted(df_manifest["fold"].dropna().unique().astype(int))}')
print(f'  pdl_norm:       [{pdl_norm_valid.min():.3f}, {pdl_norm_valid.max():.3f}]')
print('═'*55)

Validando contrato del manifest...
═══════════════════════════════════════════════════════
✅ CONTRATO DEL MANIFEST VERIFICADO
═══════════════════════════════════════════════════════
  Filas:          2,433
  Match PDL:      92.9%
  Cell lines:     ['hFB1', 'hFB10', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2', 'hFB5', 'hFB6', 'hFB7', 'hFB8', 'hFB9']
  Folds:          [0, 1, 2]
  pdl_norm:       [0.047, 1.000]
═══════════════════════════════════════════════════════


## 11. Resumen final y checklist de calidad

In [26]:
# ══ SECCIÓN 11 — Resumen final ══════════════════════════════════════════════

print('='*60)
print('RESUMEN DEL MANIFEST')
print('='*60)
print(f'Imágenes totales parseadas: {len(df_manifest):,}')
print(f'Imágenes con PDL:           {df_manifest["has_pdl"].sum():,}')
print(f'Fallas de parser:           {len(df_failed):,}')
print(f'Imágenes excluidas:         {excluded:,}')
print()
print('Por fase:')
print(df_manifest.groupby('phase').agg(n=('img_path','count'), con_pdl=('has_pdl','sum')))
print()
print('CHECKLIST MVP-1')
print('-'*40)
checks = [
    ('PRETRAIN  ≥ 1,500 imágenes',     len(mvp1_pretrain) >= 1500),
    ('FINETUNE  ≥ 300 imágenes',       len(mvp1_finetune) >= 300),
    ('EVAL      ≥ 150 imágenes',       len(mvp1_eval) >= 150),
    ('PDL disponible en >80%',         df_manifest['has_pdl'].mean() > 0.8),
    ('Fallas parser <5%',              len(df_failed)/max(len(df_manifest),1) < 0.05),
    ('Splits asignados',               'fold' in df_manifest.columns),
    ('Contrato verificado',            True),  # si llegamos aquí, el contrato pasó
]
for desc, passed in checks:
    print(f'  {"✅" if passed else "❌"} {desc}')
print()
print('Archivos generados:')
print(f'  {out_full}')
print(f'  {out_pretrain}')
print(f'  {out_finetune}')
print(f'  {out_eval}')
print(f'  {meta_path}')

RESUMEN DEL MANIFEST
Imágenes totales parseadas: 2,433
Imágenes con PDL:           2,261
Fallas de parser:           2
Imágenes excluidas:         832

Por fase:
             n  con_pdl
phase                  
PhaseI     425      425
PhaseII   1217     1073
PhaseIII   162      162
PhaseV     629      601

CHECKLIST MVP-1
----------------------------------------
  ✅ PRETRAIN  ≥ 1,500 imágenes
  ✅ FINETUNE  ≥ 300 imágenes
  ✅ EVAL      ≥ 150 imágenes
  ✅ PDL disponible en >80%
  ✅ Fallas parser <5%
  ✅ Splits asignados
  ✅ Contrato verificado

Archivos generados:
  /Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS/manifest_full_20260327_152829.csv
  /Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS/manifest_mvp1_pretrain_20260327_152829.csv
  /Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS/manifest_mvp1_finetune_20260327_152829.csv
  /Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS/manifest_mvp1_eval_20260327_152829.csv
  /Users/JCB/Documentos/Proyecto Inte